# 4 Generalization

In [10]:
import pandas as pd
import joblib
import numpy as np
# step 1: Load test data
test_df = pd.read_csv('test_data.csv')
test_df.columns = test_df.columns.str.strip().str.replace(' ', '_')

# Later: Read features
with open('selected_features_k_means.txt', 'r') as f:
    selected_features = [line.strip() for line in f]

# Predict cluster labels

In [11]:


# Save Index for final output
test_index = test_df['Index']

# Drop unnecessary columns (only features remain)
X_test_for_cluster=test_df[selected_features]

# Step 2: Load scaler and KMeans model used for clustering
scaler_k_means = joblib.load('scaler.pkl')
kmeans = joblib.load('kmeans_model.pkl')

# Step 3: Apply scaling for clustering
X_test_scaled_for_cluster = scaler_k_means.transform(X_test_for_cluster)

# Step 4: Predict ClusterID for each test company
cluster_labels = kmeans.predict(X_test_scaled_for_cluster)
test_df['ClusterID'] = cluster_labels
length_cluster=[len(test_df[test_df['ClusterID']==i]) for i in range(9)]
print(length_cluster)

[223, 82, 285, 0, 0, 1, 61, 2, 358]


#  Predict each company's bankruptcy status in the test data

In [12]:




# Step 5: Prepare a DataFrame to collect final predictions
final_predictions = pd.DataFrame({'Index': test_index})

# Step 6: Define constant clusters manually
# (This list should come from your train data analysis)
constant_predict_zero = [1]  # Clusters with only non-bankrupt companies (ĥ = 0)


# Step 7: Loop through each unique ClusterID
for i in test_df['ClusterID'].unique():

    # Get all rows belonging to this cluster
    subgroup_df = test_df[test_df['ClusterID'] == i]

    if len(subgroup_df) == 0:
        continue  # skip empty clusters

    # Step 7.1: Handle constant clusters
    if id in constant_predict_zero:
        final_predictions.loc[subgroup_df.index, 'Bankrupt?'] = 0
        continue


    # Step 7.2: Handle normal clusters (load scaler + stacking model)


    try:

        stacked_model = joblib.load(f'stacked_model_cluster{i}.pkl')

    except:
        # If model missing by mistake, default to safe prediction (non-bankrupt)
        final_predictions.loc[subgroup_df.index, 'Bankrupt?'] = 0
        continue

    # Step 7.3: Prepare features for this subgroup
    X_subgroup=pd.DataFrame()


    try:

        with open(f'selected_features_cluster{i}.txt', 'r') as f:

            selected_features = [line.strip() for line in f]
            X_subgroup = subgroup_df[selected_features]

    except:
        X_subgroup = subgroup_df.drop(columns=['Index', 'ClusterID'])
        continue

    try:

        scaler_subgroup = joblib.load(f'scaler_cluster{i}.pkl')
        # Step 7.4: Apply subgroup-specific scaling
        X_subgroup = scaler_subgroup.transform(X_subgroup)
    except:
        continue

    # Step 7.5: Predict using stacking model
    y_pred_subgroup = stacked_model.predict(X_subgroup)
    print(f'Predicted bancruptcy of cluster {i}: {np.sum(y_pred_subgroup)}')

    # Step 7.6: Save predictions into final DataFrame
    final_predictions.loc[subgroup_df.index, 'Bankrupt?'] = y_pred_subgroup

# Step 8: Save final submission
final_predictions['Bankrupt?'] = final_predictions['Bankrupt?'].fillna(0)
final_predictions = final_predictions.astype({'Bankrupt?': 'int'})
final_predictions.to_csv('3_Generalization.csv', index=False)

print("Final prediction file saved: 3_Generalization.csv\n")


Predicted bancruptcy of cluster 8: 52
Predicted bancruptcy of cluster 2: 19
Predicted bancruptcy of cluster 6: 12
Predicted bancruptcy of cluster 7: 0
Predicted bancruptcy of cluster 0: 7
Predicted bancruptcy of cluster 5: 0
Final prediction file saved: 3_Generalization.csv



In [13]:
df=pd.read_csv('3_Generalization.csv')
# Get the sum of the 'Bankrupt?' column
total_bankrupt = df['Bankrupt?'].sum()
print(f"Total number of bankrupt companies predicted: {total_bankrupt}")

Total number of bankrupt companies predicted: 90
